# The Greenhouse With Flaky Sensors

**Topic:** Vectorized NumPy &nbsp;·&nbsp; **Difficulty:** Easy

A research greenhouse logs six sensors once a minute. The sensors are cheap, so
roughly one reading in twenty comes back as `NaN`, and one sensor spends the
whole month reporting in a completely different unit. Your job is the boring part
nobody wants: clean the matrix before anyone fits a model to it.

The catch is that a single Python loop over 40,000 readings is a needless
bottleneck, and writing it vectorized is exactly the skill this module teaches.

## The data (created in the next cell)

- `readings` `(40000, 6)`: sensor readings, with `NaN` scattered through them.
- Column 3 is recorded in a different unit from the rest, so its raw scale
  means nothing.
- No column is entirely `NaN`.

## Your task

With **NumPy only** (no `pandas`, no loops over rows):

1. Compute each column's mean and standard deviation **ignoring** the `NaN`s
   (population values, i.e. `np.nanmean` and `np.nanstd` with their default
   ddof=0).
2. Replace every `NaN` with its own column's mean.
3. Standardize each column with the statistics from step 1 (the ones computed
   before imputation).
4. Set `answers` to a dict with keys `"col_means"`, `"col_stds"` (both length-6
   arrays from step 1) and `"cleaned"` (the `(40000, 6)` standardized matrix).

## Evaluation

For each of the three arrays, the grader checks two scalars derived from it,
its total and a fixed random projection, against stored reference values to a
relative tolerance of 1e-6.


In [ ]:
import numpy as np

rng = np.random.default_rng(11)
n = 40000
base = rng.normal([20.0, 55.0, 1013.0, 0.0, 7.2, 400.0],
                  [2.0, 8.0, 5.0, 1.0, 0.4, 60.0], (n, 6))
base[:, 3] = rng.normal(300.0, 90.0, n)      # the odd-unit column
mask = rng.random((n, 6)) < 0.05
readings = base.copy()
readings[mask] = np.nan
print(f"readings {readings.shape}, missing: {np.isnan(readings).sum()}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the data created above to solve the task, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("col_means", [6], 1795.0708642835239, -714.0717398020886, 1e-06),
    ("col_stds", [6], 165.05308369524522, 184.83536481114598, 1e-06),
    ("cleaned", [40000, 6], -5.49147216588608e-09, 155.8231658215873, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

`np.nanmean` and `np.nanstd` skip the gaps for you. Impute with
`np.where(np.isnan(...))`, then standardize with the *pre-imputation*
statistics. Imputing first and recomputing would shrink every standard
deviation toward zero.

```python
import numpy as np

col_means = np.nanmean(readings, axis=0)
col_stds = np.nanstd(readings, axis=0)

filled = np.where(np.isnan(readings), col_means, readings)
cleaned = (filled - col_means) / col_stds

answers = {"col_means": col_means, "col_stds": col_stds, "cleaned": cleaned}
```